# 3SBB — model-vs-experiment correlation review (v4 calibration)

This notebook reviews the v4 calibration of the reduced-order 3SBB model against
the IQS experimental FRF dataset.

It works on a **per-case median dataset** (`experimental_frfs_median.h5`, ~12 MB,
committed in this repo), produced once by collapsing the 2 638-measurement
archive to one summary FRF per experimental case.  No Drive mount or external
download is required for normal use.  Section 1 also explains how to **rebuild
the median file from the full archive** if you ever need to.

What v4 changes vs the previous calibration:

* per-storey joint stiffness ratio (3 × JSR) and per-floor extra mass (3 × m)
* heavier base attachment mass allowed (≤ 60 kg)
* per-mode damping for the 3 Y-modes
* **objective driven by FRF correlation** (Pearson on log-magnitude + complex
  cosine similarity) plus frequency residuals + mode-shape MAC across the
  6 upper-floor Y sensors


## 1. Setup

Clone the branch and install dependencies.

In [ ]:
import os, subprocess, sys
BRANCH = 'claude/optimize-model-correlations-QDOhT'
REPO   = 'https://github.com/grcarmenaty/PhD_LANL.git'
if not os.path.isdir('/content/PhD_LANL'):
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO, '/content/PhD_LANL'], check=True)
%cd /content/PhD_LANL
!pip install -q -r requirements.txt gdown


### 1a. (one-time) Build `experimental_frfs_median.h5`

Skip this section if `experimental_frfs_median.h5` is already in the repo —
that's the normal case. Run it only when:

* you're setting up the dataset for the first time, or
* the upstream IQS archive has been updated.

The cell below downloads the 305 MB archive from the public Drive link and
collapses it to per-case medians.  ETA on Colab: ~1 min download + ~30 s
median computation.  The resulting file is ~12 MB and is committed back to
this branch via the `git push` cell that follows.

In [ ]:
from pathlib import Path
MEDIAN = Path('/content/PhD_LANL/experimental_frfs_median.h5')
if MEDIAN.exists():
    print('Median file already present — skip the build.')
else:
    !python build_median_dataset.py
    print('Built:', MEDIAN, f'({MEDIAN.stat().st_size/1e6:.2f} MB)')


In [ ]:
# OPTIONAL — push the freshly built median file to GitHub so future runs skip
# the build entirely.  Requires a fine-grained PAT with `Contents: write` on
# grcarmenaty/PhD_LANL.  Replace <TOKEN> below with your token, or skip.
#
# !git config user.email 'you@example.com'
# !git config user.name  'you'
# !git remote set-url origin https://<TOKEN>@github.com/grcarmenaty/PhD_LANL.git
# !git add experimental_frfs_median.h5 && git commit -m 'Add per-case median FRF dataset'
# !git push


## 2. Load the median dataset

In [ ]:
import numpy as np, h5py, sys
sys.path.insert(0, '/content/PhD_LANL')
from reduced_model_semirigid import (BuildingGeometry, modes,
                                       compute_frf_matrix, point_to_dof_vector)

with h5py.File('/content/PhD_LANL/experimental_frfs_median.h5', 'r') as f:
    freq_exp       = np.array(f['freq'])
    mag_median     = np.array(f['mag_median'])      # (n_cases, 1601, 9) float32
    complex_median = np.array(f['complex_median'])  # (n_cases, 1601, 9) complex64
    case_names     = [c.decode() if isinstance(c, bytes) else str(c)
                      for c in np.array(f['case_names'])]
    case_counts    = np.array(f['case_counts'])

print(f'Cases:               {len(case_names)}')
print(f'Total measurements:  {int(case_counts.sum())}')
print(f'Top 5 by count:')
for nm, ct in sorted(zip(case_names, case_counts), key=lambda x: -x[1])[:5]:
    print(f'  {ct:4d}  {nm}')

ci_pristine = case_names.index('Pristine')
H_mean  = mag_median[ci_pristine]       # (1601, 9)
H_cmean = complex_median[ci_pristine]   # (1601, 9)


In [ ]:
# Frequency-band setup
FLOOR_Y_CHS = [1, 2, 3, 5, 6, 7]   # S5,S6,S7,S11,S12,S13
F_LO, F_HI  = 10.0, 80.0
mask_band   = (freq_exp >= F_LO) & (freq_exp <= F_HI)
freq_band   = freq_exp[mask_band]

# Identify Y-mode resonances on S7 (ch3)
from scipy.signal import find_peaks
ch3 = H_mean[mask_band, 3]
peaks, props = find_peaks(ch3, distance=40, prominence=ch3.max()*0.05)
top3  = peaks[np.argsort(props['prominences'])[-3:]]
top3  = top3[np.argsort(freq_band[top3])]
F_EXP = freq_band[top3]
print('Y-mode targets:', F_EXP, 'Hz')


## 3. Baseline (v3-style calibration) for comparison

In [ ]:
# v3 fixed parameters
geom_v3 = BuildingGeometry(joint_stiffness_ratio=7.9, damping=0.005, base_extra_mass=6.5)
geom_v3.column_factor = np.array([[1.23]*4, [1.02]*4, [1.14]*4])

# Sensor / shaker positions (must match calibrate_3sbb_v4.py)
cx = geom_v3.plate_lx/2; PL = geom_v3.plate_ly
xl = geom_v3.col_lx/2;   xh = geom_v3.plate_lx - xl
z0 = geom_v3.plate_z_centre(0)
z1 = geom_v3.plate_z_centre(1); z2 = geom_v3.plate_z_centre(2); z3 = geom_v3.plate_z_centre(3)
INPUT_PT   = ((cx, 0., z0), 'Y')
OUTPUT_PTS = [
    ((xl, PL, z3), 'Y'), ((xl, PL, z2), 'Y'), ((xl, PL, z1), 'Y'),
    ((xh, PL, z3), 'Y'), ((xh, PL, z2), 'Y'), ((xh, PL, z1), 'Y'),
]

def y_modes_with_shape(geom, output_pts):
    f_all, V, _ = modes(geom)
    b = point_to_dof_vector((cx, 0., z0), 'Y', geom); bphi = np.abs(V.T @ b)
    ie = np.where(f_all > 0.5)[0]
    order = np.argsort(-bphi[ie])
    y_idx = ie[order[:3]]; y_idx = y_idx[np.argsort(f_all[y_idx])]
    C = np.stack([point_to_dof_vector(p, d, geom) for p, d in output_pts], axis=1)
    return f_all[y_idx], (V.T @ C)[y_idx]


def frf_correlations(geom, damp_arr):
    H = compute_frf_matrix(freq_band, [INPUT_PT], OUTPUT_PTS, geom, damping=damp_arr)[:, :, 0]
    H_exp = H_cmean[mask_band][:, FLOOR_Y_CHS]
    log_m = np.log10(np.abs(H) + 1e-12)
    log_e = np.log10(np.abs(H_exp) + 1e-12)
    Lc = log_m - log_m.mean(0); Le = log_e - log_e.mean(0)
    pearson = np.sum(Lc*Le, axis=0) / ((np.linalg.norm(Lc, axis=0)+1e-30) *
                                        (np.linalg.norm(Le, axis=0)+1e-30))
    Hn  = H/(np.linalg.norm(H,  axis=0, keepdims=True)+1e-30)
    Hen = H_exp/(np.linalg.norm(H_exp, axis=0, keepdims=True)+1e-30)
    cos_complex = np.real(np.sum(Hn*Hen.conj(), axis=0))
    return pearson, cos_complex, H, H_exp

f_v3, _ = y_modes_with_shape(geom_v3, OUTPUT_PTS)
zeta_v3 = np.array([0.005, 0.006, 0.005, 0.005, 0.005, 0.005, 0.005, 0.005, 0.005])
pear_v3, cos_v3, H_v3, H_exp = frf_correlations(geom_v3, zeta_v3[:9])
print(f'v3 Y-modes:        {f_v3}')
print(f'v3 freq err (%):   {[(f_v3[j]-F_EXP[j])/F_EXP[j]*100 for j in range(3)]}')
print(f'v3 mean Pearson:   {pear_v3.mean():.4f}')
print(f'v3 mean cosine:    {cos_v3.mean():.4f}')


## 4. Run v4 calibration

`calibrate_3sbb_v4.py` auto-detects the median dataset and uses it directly.

In [ ]:
!python calibrate_3sbb_v4.py 2>&1 | tail -50

## 5. Load v4 calibration & compute new correlations

In [ ]:
cal = np.load('/content/PhD_LANL/calibration_result.npz')
print('Saved fields:', cal.files)

geom_v4 = BuildingGeometry(
    joint_stiffness_ratio = float(cal['jsr']),
    joint_stiffness_ratio_per_storey = np.array(cal['jsr_per_storey']).tolist(),
    base_extra_mass = float(cal['base_extra_mass']),
    floor_extra_mass = np.array(cal['floor_extra_mass']).tolist(),
    damping = float(cal['damping']),
)
geom_v4.column_factor = np.array([[float(cal['cf_s1'])]*4,
                                   [float(cal['cf_s2'])]*4,
                                   [float(cal['cf_s3'])]*4])
zeta_y = np.array(cal['zeta_y_modes'])

# Build per-mode damping array, Y-modes ranked by participation
f_all, V, _ = modes(geom_v4)
b = point_to_dof_vector((cx, 0., z0), 'Y', geom_v4); bphi = np.abs(V.T @ b)
ie = np.where(f_all > 0.5)[0]
order = np.argsort(-bphi[ie]); y_idx = ie[order[:3]]; y_idx = y_idx[np.argsort(f_all[y_idx])]
zeta_full = np.full(len(ie), float(zeta_y.mean()))
for rank, idx in enumerate(y_idx):
    zeta_full[int(np.where(ie==idx)[0][0])] = zeta_y[rank]

f_v4, _ = y_modes_with_shape(geom_v4, OUTPUT_PTS)
pear_v4, cos_v4, H_v4, _ = frf_correlations(geom_v4, zeta_full)
print(f'v4 Y-modes:        {f_v4}')
print(f'v4 freq err (%):   {[(f_v4[j]-F_EXP[j])/F_EXP[j]*100 for j in range(3)]}')
print(f'v4 mean Pearson:   {pear_v4.mean():.4f}')
print(f'v4 mean cosine:    {cos_v4.mean():.4f}')


## 6. Side-by-side correlation comparison

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
sensor_lbls = ['S5','S6','S7','S11','S12','S13']
xx = np.arange(len(sensor_lbls)); w = 0.35

axes[0].bar(xx-w/2, pear_v3, w, label='v3', color='steelblue')
axes[0].bar(xx+w/2, pear_v4, w, label='v4 (new)', color='tab:green')
axes[0].set_xticks(xx); axes[0].set_xticklabels(sensor_lbls)
axes[0].set_ylim(0, 1); axes[0].grid(axis='y', alpha=0.3)
axes[0].set_title('Pearson corr (log|H|) — model vs experiment'); axes[0].legend()

axes[1].bar(xx-w/2, cos_v3, w, label='v3', color='steelblue')
axes[1].bar(xx+w/2, cos_v4, w, label='v4 (new)', color='tab:green')
axes[1].set_xticks(xx); axes[1].set_xticklabels(sensor_lbls)
axes[1].set_ylim(0, 1); axes[1].grid(axis='y', alpha=0.3)
axes[1].set_title('Complex cosine similarity — model vs experiment'); axes[1].legend()
plt.tight_layout(); plt.show()

print(f'v3 mean Pearson = {pear_v3.mean():.4f}    v4 mean Pearson = {pear_v4.mean():.4f}'
      f'    Δ = {pear_v4.mean()-pear_v3.mean():+.4f}')
print(f'v3 mean cosine  = {cos_v3.mean():.4f}    v4 mean cosine  = {cos_v4.mean():.4f}'
      f'    Δ = {cos_v4.mean()-cos_v3.mean():+.4f}')


## 7. FRF overlay — model vs experiment (6 floor sensors)

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(13, 10), sharex=True)
for ci, lbl in enumerate(sensor_lbls):
    r, c = ci % 3, ci // 3
    ax = axes[r, c]
    ax.semilogy(freq_band, np.abs(H_exp[:, ci]),    'k',  lw=1.4, label='Experiment')
    ax.semilogy(freq_band, np.abs(H_v3[:,  ci]), 'b--', lw=1.0, label='Model v3')
    ax.semilogy(freq_band, np.abs(H_v4[:,  ci]), 'r-',  lw=1.4, label='Model v4')
    ax.set_title(lbl); ax.grid(True, which='both', ls=':', alpha=0.4); ax.set_xlim(F_LO, F_HI)
    if r == 2: ax.set_xlabel('Frequency (Hz)')
    if c == 0: ax.set_ylabel('|H|  (m/s²)/N')
    if r==0 and c==0: ax.legend(fontsize=8)
plt.suptitle('Reduced-order model vs experiment FRF — v3 vs v4 calibration', y=1.005)
plt.tight_layout(); plt.show()


## 8. CFDAC — model vs experiment median

In [ ]:
def cfdac(H1, H2):
    R1 = H1 / (np.linalg.norm(H1, axis=1, keepdims=True) + 1e-30)
    R2 = H2 / (np.linalg.norm(H2, axis=1, keepdims=True) + 1e-30)
    return np.abs(R1 @ R2.conj().T) ** 2

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
for ax, (M, ttl) in zip(axes, [(H_v3, 'v3 model vs exp(median)'),
                                 (H_v4, 'v4 model vs exp(median)')]):
    C = cfdac(M, H_exp)
    im = ax.imshow(C, vmin=0, vmax=1, cmap='jet', aspect='auto',
                   extent=[F_LO, F_HI, F_HI, F_LO])
    ax.set_title(ttl); ax.set_xlabel('Freq exp (Hz)')
axes[0].set_ylabel('Freq model (Hz)')
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.7, label='CFDAC')
plt.show()


## 9. Damage scenarios — model vs experiment

A quick sanity check: how well does the calibrated model track the
experimental FRFs for *damaged* cases?  We pick a few representative cases
(plus pristine) and overlay the predicted FRF on the experimental median.

In [ ]:
import sys
sys.path.insert(0, '/content/PhD_LANL')
from damage_scenarios import geometry_for_case

show_cases = ['Pristine', 'D (11%) 1BD', 'D(50%) 1BD', 'Damage (85%) 1BD']

fig, axes = plt.subplots(len(show_cases), 1, figsize=(11, 9), sharex=True)
SHOW_CH = 2   # S7 — fl1 Y, all 3 modes visible
for r, cn in enumerate(show_cases):
    ax = axes[r]
    # experimental median for this case
    if cn in case_names:
        ci = case_names.index(cn)
        H_e = mag_median[ci, mask_band, FLOOR_Y_CHS[SHOW_CH]]
        ax.semilogy(freq_band, H_e, 'k', lw=1.4, label='exp median')
    # model
    g_m = geometry_for_case(cn)
    H_m = compute_frf_matrix(freq_band, [INPUT_PT], OUTPUT_PTS, g_m,
                              damping=zeta_full)[:, SHOW_CH, 0]
    ax.semilogy(freq_band, np.abs(H_m), 'r', lw=1.2, label='v4 model')
    ax.set_title(cn); ax.grid(True, which='both', ls=':', alpha=0.4)
    ax.set_xlim(F_LO, F_HI); ax.set_ylabel('|H| @ S7')
    if r == 0: ax.legend(fontsize=8)
axes[-1].set_xlabel('Frequency (Hz)')
plt.suptitle('Damage scenarios — sensor S7', y=1.002)
plt.tight_layout(); plt.show()


## Summary

The v4 calibration uses per-storey JSR + per-floor mass + a heavier base
attachment to break the uniform-Y-chain frequency-ratio constraint, and
optimises directly against FRF correlation rather than just RMS magnitude.
The bar charts in section 6 show the per-channel Pearson and complex
cosine similarity gains over the v3 baseline.

The slim per-case median dataset (~12 MB) lives in this repo as
`experimental_frfs_median.h5`; the 305 MB raw archive is no longer needed
for any of the analysis above.
